<a href="https://colab.research.google.com/github/manya28/TransformerJM/blob/main/LME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# install.packages(c("nlme", "JMbayes2", "survival", "lattice"))

# Load necessary libraries
library(nlme)       # For Linear Mixed Effects Model (LME)
library(JMbayes2)         # For Joint Modelling
library(survival)   # For Cox model
library(dplyr)  # or library(magrittr)

# Read CSV files into R
train_data <- read.csv("train_data.csv")
test_data  <- read.csv("test_data.csv")
val_data   <- read.csv("val_data.csv")

# Check structure of the data
str(train_data)
train_data <- subset(train_data,select = -X)
head(train_data)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message in install.packages(c("nlme", "JMbayes2", "survival", "lattice")):
“installation of package ‘survival’ had non-zero exit status”


'data.frame':	16251 obs. of  16 variables:
 $ X              : int  0 1 15 17 18 19 20 21 22 60 ...
 $ pid            : int  200143 200551 200701 200800 200817 201005 201005 201005 201005 201519 ...
 $ base_age       : num  7.22 11.21 7.13 7.24 16.13 ...
 $ sex            : num  1 0 0 0 0 0 0 0 0 0 ...
 $ tstart         : num  0 0 0 0 0 ...
 $ tstop          : num  0.7556 0.9829 0.0192 0.4298 0.0438 ...
 $ t1d            : int  0 1 1 1 1 0 0 0 0 0 ...
 $ aabStatus      : int  1 0 1 1 1 1 1 1 1 0 ...
 $ c_pep_auc      : num  7.11 2.31 3.71 2.86 6.95 ...
 $ hba1c          : num  4.9 6 5.4 5.6 6.1 4.8 4.8 4.8 5 5 ...
 $ fasting_glucose: num  103 97 94 87 123 81 81 81 88 78 ...
 $ early_pep_delta: num  7.27 0.87 2.31 2.54 2.12 ...
 $ astart         : num  7.22 11.21 7.13 7.24 16.13 ...
 $ astop          : num  7.97 12.19 7.15 7.67 16.17 ...
 $ glu_2hr        : num  121 308 245 192 290 108 108 108 98 112 ...
 $ time_int       : num  0.7556 0.9829 0.0192 0.4298 0.0438 ...


,pid,base_age,sex,tstart,tstop,t1d,aabStatus,c_pep_auc,hba1c,fasting_glucose,early_pep_delta,astart,astop,glu_2hr,time_int
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,200143,7.216290,1,0,0.75564682,0,1,7.114583,4.9,103,7.275,7.216290,7.971937,121,0.75564682
2,200551,11.205339,0,0,0.98288843,1,0,2.313333,6.0,97,0.870,11.205339,12.188227,308,0.98288843
3,200701,7.128679,0,0,0.01916496,1,1,3.712917,5.4,94,2.315,7.128679,7.147844,245,0.01916496
4,200800,7.240931,0,0,0.42984257,1,1,2.858750,5.6,87,2.545,7.240931,7.670773,192,0.42984257
5,200817,16.128679,0,0,0.04380561,1,1,6.952917,6.1,123,2.125,16.128679,16.172485,290,0.04380561
6,201005,18.251882,0,0,0.74469541,0,1,5.190417,4.8,81,3.175,18.251882,18.996578,108,0.74469541


In [7]:
surv_data <- train_data %>%
  group_by(pid) %>%
  summarise(
    base_age = first(base_age),
    sex = first(sex),
    aabStatus = first(aabStatus),
    tstop = max(as.numeric(tstop)),   # using the maximum 'tstop' as the follow-up time
    t1d = max(t1d)                     # event indicator should be consistent across rows per patient
  )

In [8]:
surv_fit <- coxph(Surv(tstop, t1d) ~ base_age + aabStatus,
                  data = surv_data,
                  x = TRUE)

In [9]:
surv_fit

Call:
coxph(formula = Surv(tstop, t1d) ~ base_age + aabStatus, data = surv_data, 
    x = TRUE)

               coef exp(coef)  se(coef)      z        p
base_age  -0.016520  0.983616  0.004665 -3.541 0.000399
aabStatus  2.126192  8.382883  0.151413 14.042  < 2e-16

Likelihood ratio test=360.1  on 2 df, p=< 2.2e-16
n= 2687, number of events= 428 

In [10]:
lme_med <- lme(glu_2hr ~ astart + base_age + hba1c + fasting_glucose + aabStatus,
                  random = ~ tstart | pid,
                  data = train_data)

# Longitudinal model for c_pep_auc (also known to affect T1D staging)
lme_large_1 <- lme(c_pep_auc ~ astart + hba1c + fasting_glucose + aabStatus + glu_2hr,
                random = ~ tstart | pid,
                data = train_data)

lme_large_2 <- lme(early_pep_delta ~ astart + hba1c + fasting_glucose + aabStatus + glu_2hr,
                    random = ~ tstart | pid,
                    data = train_data)

In [14]:
# Increase the number of iterations, burn-in, and thinning in the jm() function.
joint_model_med <- jm(surv_fit, lme_med, time_var = "tstop",
                     control = list(n_iter = 1000, n_burnin = 500, n_thin = 10))
   # This gives the model more chances to converge.

# Check the summary of the joint model. The association parameters (often denoted alpha)
# will indicate the strength of each biomarker's effect on the hazard of T1D onset.
joint_model_large_cpep <- jm(surv_fit, lme_large_1, time_var = "tstop",
                     control = list(n_iter = 1000, n_burnin = 500, n_thin = 10))

joint_model_large_auc <- jm(surv_fit, lme_large_2, time_var = "tstop",
                     control = list(n_iter = 1000, n_burnin = 500, n_thin = 10))

In [13]:
summary(joint_model_med)


Call:
jm(Surv_object = surv_fit, Mixed_objects = lme_med, time_var = "tstop", 
    control = list(n_iter = 1000, n_burnin = 500, n_thin = 10))

Data Descriptives:
Number of Groups: 2687		Number of events: 428 (15.9%)
Number of Observations:
  glu_2hr: 16251

                 DIC     WAIC      LPML
marginal    148634.6 148613.9 -74306.53
conditional 187993.3 186426.1 -94593.24

Random-effects covariance matrix:
                      
       StdDev    Corr 
(Intr) 38.0766 (Intr) 
tstart 7.7160  -0.8344

Survival Outcome:
                  Mean  StDev    2.5%   97.5% P   Rhat
base_age       -0.0322 0.0064 -0.0436 -0.0201 0 0.9988
aabStatus       1.8406 0.1620  1.4831  2.1561 0 1.0049
value(glu_2hr)  0.0347 0.0014  0.0327  0.0374 0 1.2172

Longitudinal Outcome: glu_2hr (family = gaussian, link = identity)
                   Mean  StDev    2.5%   97.5%      P   Rhat
(Intercept)     13.5719 5.4367  1.9738 23.1587 0.0267 1.7513
astart          -2.8391 0.1347 -3.0489 -2.5791 0.0000 8.0141
bas

In [15]:
summary(joint_model_large_cpep)


Call:
jm(Surv_object = surv_fit, Mixed_objects = lme_large_1, time_var = "tstop", 
    control = list(n_iter = 1000, n_burnin = 500, n_thin = 10))

Data Descriptives:
Number of Groups: 2687		Number of events: 428 (15.9%)
Number of Observations:
  c_pep_auc: 16251

                 DIC     WAIC      LPML
marginal    59555.37 59454.41 -29728.79
conditional 68717.38 67171.79 -34999.90

Random-effects covariance matrix:
                     
       StdDev   Corr 
(Intr) 2.4791 (Intr) 
tstart 0.3339 -0.3992

Survival Outcome:
                    Mean  StDev    2.5%   97.5%    P   Rhat
base_age         -0.0012 0.0068 -0.0129  0.0122 0.88 1.0128
aabStatus         2.0577 0.1681  1.7294  2.3261 0.00 1.2642
value(c_pep_auc) -0.1792 0.0276 -0.2344 -0.1317 0.00 1.0357

Longitudinal Outcome: c_pep_auc (family = gaussian, link = identity)
                   Mean  StDev    2.5%   97.5%      P   Rhat
(Intercept)      0.2578 0.4767 -0.5728  1.2239 0.6933 2.1541
astart           0.0701 0.0072  0.0557  

In [16]:
summary(joint_model_large_auc)


Call:
jm(Surv_object = surv_fit, Mixed_objects = lme_large_2, time_var = "tstop", 
    control = list(n_iter = 1000, n_burnin = 500, n_thin = 10))

Data Descriptives:
Number of Groups: 2687		Number of events: 428 (15.9%)
Number of Observations:
  early_pep_delta: 16251

                 DIC     WAIC      LPML
marginal    62564.44 62547.32 -31271.15
conditional 72677.28 71079.28 -36937.76

Random-effects covariance matrix:
                     
       StdDev   Corr 
(Intr) 2.2309 (Intr) 
tstart 0.3263 -0.4079

Survival Outcome:
                          Mean  StDev    2.5%   97.5%    P   Rhat
base_age               -0.0050 0.0066 -0.0176  0.0085 0.44 1.1444
aabStatus               1.8521 0.1593  1.5071  2.1182 0.00 1.0499
value(early_pep_delta) -0.5354 0.0435 -0.6231 -0.4583 0.00 1.0453

Longitudinal Outcome: early_pep_delta (family = gaussian, link = identity)
                   Mean  StDev    2.5%   97.5%      P   Rhat
(Intercept)      3.4693 0.3901  2.7117  4.2521 0.0000 1.0473
asta

In [18]:
# Organize the longitudinal models in a list.
lme_list <- list(
  glu_2hr = lme_med,
  c_pep_auc = lme_large_1,
  early_pep_delta = lme_large_2
)

In [19]:
joint_model <- jm(surv_fit, lme_list, time_var = "tstop",
                  control = list(n_iter = 1000, n_burnin = 500, n_thin = 10))

In [20]:
summary(joint_model)


Call:
jm(Surv_object = surv_fit, Mixed_objects = lme_list, time_var = "tstop", 
    control = list(n_iter = 1000, n_burnin = 500, n_thin = 10))

Data Descriptives:
Number of Groups: 2687		Number of events: 428 (15.9%)
Number of Observations:
  glu_2hr: 16251
  c_pep_auc: 16251
  early_pep_delta: 16251

                 DIC     WAIC      LPML
marginal    260921.2 260957.2 -130480.7
conditional 313012.8 308526.9 -156730.9

Random-effects covariance matrix:
                                                      
       StdDev    Corr                                 
(Intr) 38.1699 (Intr)  tstart  (Intr)  tstart  (Intr) 
tstart 7.7469  -0.8363                                
(Intr) 2.5312  -0.2166 0.2015                         
tstart 0.3685  -0.0227 0.0364  -0.3754                
(Intr) 2.3273  -0.2199 0.1536  0.9203  -0.3779        
tstart 0.3680  -0.0307 -0.0027 -0.4140 0.9353  -0.4118

Survival Outcome:
                          Mean  StDev    2.5%   97.5% P   Rhat
base_age          